## Part 1: Install Java and Download Kafka

In [1]:
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!java -version

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)


In [2]:
KAFKA_VERSION = "3.9.1"
SCALA_VERSION = "2.13"

import os

primary_url = f"https://dlcdn.apache.org/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz"
fallback_url = f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz"

!wget -q "{primary_url}" -O kafka.tgz

size = os.path.getsize("kafka.tgz") if os.path.exists("kafka.tgz") else 0
print(f"Downloaded kafka.tgz: {size} bytes")

if size < 1_000_000:  # a real Kafka tgz is tens of MB; anything under 1MB is a failed download
    print("Primary download failed or too small -- retrying with archive.apache.org fallback...")
    !wget -q "{fallback_url}" -O kafka.tgz
    size = os.path.getsize("kafka.tgz") if os.path.exists("kafka.tgz") else 0
    print(f"Downloaded kafka.tgz (fallback): {size} bytes")

assert size > 1_000_000, (
    "Kafka download still failed after fallback. Check your Colab internet "
    "connection, or manually verify the URL at https://kafka.apache.org/downloads "
    "and update KAFKA_VERSION above."
)

!tar -xzf kafka.tgz
!mv kafka_{SCALA_VERSION}-{KAFKA_VERSION} kafka
!ls kafka/bin | head -10

Downloaded kafka.tgz: 0 bytes
Primary download failed or too small -- retrying with archive.apache.org fallback...
Downloaded kafka.tgz (fallback): 122110298 bytes
mv: cannot move 'kafka_2.13-3.9.1' to 'kafka/kafka_2.13-3.9.1': Directory not empty
connect-distributed.sh
connect-mirror-maker.sh
connect-plugin-path.sh
connect-standalone.sh
kafka-acls.sh
kafka-broker-api-versions.sh
kafka-client-metrics.sh
kafka-cluster.sh
kafka-configs.sh
kafka-console-consumer.sh


## Part 2: Start Kafka in KRaft Mode (No ZooKeeper)

In [23]:
import uuid

CLUSTER_ID = str(uuid.uuid4())
print("Cluster ID:", CLUSTER_ID)

!kafka/bin/kafka-storage.sh format -t {CLUSTER_ID} -c kafka/config/kraft/server.properties --ignore-formatted

Cluster ID: 7fc53647-5b62-4bf8-bcd2-bddaf9eff877
Exception in thread "main" java.lang.RuntimeException: Invalid cluster.id in: /tmp/kraft-combined-logs/meta.properties. Expected 7fc53647-5b62-4bf8-bcd2-bddaf9eff877, but read 9783c4b7-511a-4701-b09f-0958d55ea180
	at org.apache.kafka.metadata.properties.MetaPropertiesEnsemble.verify(MetaPropertiesEnsemble.java:503)
	at org.apache.kafka.metadata.storage.Formatter.doFormat(Formatter.java:391)
	at org.apache.kafka.metadata.storage.Formatter.run(Formatter.java:253)
	at kafka.tools.StorageTool$.runFormatCommand(StorageTool.scala:150)
	at kafka.tools.StorageTool$.execute(StorageTool.scala:86)
	at kafka.tools.StorageTool$.main(StorageTool.scala:46)
	at kafka.tools.StorageTool.main(StorageTool.scala)


In [4]:
!ls -la

total 126700
drwxr-xr-x 1 root root      4096 Jun 22 06:18 .
drwxr-xr-x 1 root root      4096 Jun 22 05:11 ..
-rw-r--r-- 1 root root       113 Jun 22 05:39 batch_vs_streaming_comparison.csv
-rw-r--r-- 1 root root   2538674 Jun 22 05:42 classified_reviews_for_elasticsearch.csv
-rw-r--r-- 1 root root   4560124 Jun 22 05:32 cleaned_data.csv
drwxr-xr-x 4 root root      4096 Jun  4 13:32 .config
drwxr-xr-x 9 root root      4096 Jun 22 06:09 kafka
drwxr-xr-x 7 root root      4096 May 12  2025 kafka_2.13-3.9.1
-rw-r--r-- 1 root root     63246 Jun 22 06:14 kafka_log.txt
-rw-r--r-- 1 root root 122110298 May 19  2025 kafka.tgz
-rw-r--r-- 1 root root    424150 Jun 22 05:33 naive_bayes_model.pickle
drwxr-xr-x 1 root root      4096 Jun  4 13:32 sample_data


In [24]:
!ls -la kafka.tgz

-rw-r--r-- 1 root root 122110298 May 19  2025 kafka.tgz


In [25]:
import subprocess

kafka_process = subprocess.Popen(
    ["kafka/bin/kafka-server-start.sh", "kafka/config/kraft/server.properties"],
    stdout=open("kafka_log.txt", "w"),
    stderr=subprocess.STDOUT
)
print(f"Kafka broker starting (PID {kafka_process.pid})... waiting 15 seconds for it to be ready.")

import time
time.sleep(15)

!tail -20 kafka_log.txt

Kafka broker starting (PID 27237)... waiting 15 seconds for it to be ready.
	zookeeper.ssl.truststore.location = null
	zookeeper.ssl.truststore.password = null
	zookeeper.ssl.truststore.type = null
 (kafka.server.KafkaConfig)
[2026-06-22 06:30:57,827] INFO [BrokerServer id=1] Waiting for the broker to be unfenced (kafka.server.BrokerServer)
[2026-06-22 06:30:57,876] INFO [BrokerLifecycleManager id=1] The broker has been unfenced. Transitioning from RECOVERY to RUNNING. (kafka.server.BrokerLifecycleManager)
[2026-06-22 06:30:57,876] INFO [ReplicaFetcherManager on broker 1] Removed fetcher for partitions Set(tng_reviews-0) (kafka.server.ReplicaFetcherManager)
[2026-06-22 06:30:57,877] INFO [BrokerServer id=1] Finished waiting for the broker to be unfenced (kafka.server.BrokerServer)
[2026-06-22 06:30:57,877] INFO authorizerStart completed for endpoint PLAINTEXT. Endpoint is now READY. (org.apache.kafka.server.network.EndpointReadyFutures)
[2026-06-22 06:30:57,878] INFO [SocketServer list

In [26]:
!kafka/bin/kafka-broker-api-versions.sh --bootstrap-server localhost:9092 2>&1 | head -5

localhost:9092 (id: 1 rack: null) -> (
	Produce(0): 0 to 11 [usable: 11],
	Fetch(1): 0 to 17 [usable: 17],
	ListOffsets(2): 0 to 9 [usable: 9],
	Metadata(3): 0 to 12 [usable: 12],


## Part 3: Create the Kafka Topic

In [27]:
TOPIC_NAME = "tng_reviews"

!kafka/bin/kafka-topics.sh --create --topic {TOPIC_NAME} --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1
!kafka/bin/kafka-topics.sh --describe --topic {TOPIC_NAME} --bootstrap-server localhost:9092

Error while executing topic command : Topic 'tng_reviews' already exists.
[2026-06-22 06:31:23,087] ERROR org.apache.kafka.common.errors.TopicExistsException: Topic 'tng_reviews' already exists.
 (org.apache.kafka.tools.TopicCommand)
Topic: tng_reviews	TopicId: P0cdw2WwSbSbecwxywGQrg	PartitionCount: 1	ReplicationFactor: 1	Configs: segment.bytes=1073741824
	Topic: tng_reviews	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 


## Part 4: Install PySpark and the Kafka Connector

In [28]:
!pip install -q pyspark==3.5.1 kafka-python

import pyspark
print("PySpark version:", pyspark.__version__)

PySpark version: 3.5.1


In [29]:
!pip install -q scikit-learn==1.9.0

## Part 5: Upload Required Files

- `cleaned_data.csv`
- `naive_bayes_model.pickle`


In [30]:
from google.colab import files
print("Select cleaned_data.csv and naive_bayes_model.pickle (you can select both at once)")
uploaded = files.upload()

Select cleaned_data.csv and naive_bayes_model.pickle (you can select both at once)


Saving cleaned_data.csv to cleaned_data (2).csv


In [31]:
from google.colab import files
print("Select cleaned_data.csv and naive_bayes_model.pickle (you can select both at once)")
uploaded = files.upload()

Select cleaned_data.csv and naive_bayes_model.pickle (you can select both at once)


Saving naive_bayes_model.pickle to naive_bayes_model (2).pickle


## Part 6: Kafka Producer — Stream Reviews into the Topic

In [32]:
from kafka import KafkaProducer
import pandas as pd
import json
import time

df = pd.read_csv('cleaned_data.csv')
print(f"Loaded {len(df)} reviews")

producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

N_STREAM_DEMO = 500
DELAY_SECONDS = 0.05   # ~20 messages/sec, fast enough to demo, slow enough to watch

sent = 0
for _, row in df.head(N_STREAM_DEMO).iterrows():
    message = {
        'review_id': str(row['review_id']),
        'cleaned_text': str(row['cleaned_text']),
        'true_sentiment': str(row['sentiment']),  # ground truth, kept for evaluation later
        'score': int(row['score']),
    }
    producer.send(TOPIC_NAME, value=message)
    sent += 1
    if sent % 100 == 0:
        print(f"Sent {sent}/{N_STREAM_DEMO}")
    time.sleep(DELAY_SECONDS)

producer.flush()
print(f"\nFinished sending {sent} reviews to topic '{TOPIC_NAME}'")

Loaded 9979 reviews
Sent 100/500
Sent 200/500
Sent 300/500
Sent 400/500
Sent 500/500

Finished sending 500 reviews to topic 'tng_reviews'


## Part 7: Spark Structured Streaming Consumer — Real-Time Classification

In [40]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1 pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import udf, col, from_json
import pickle

spark = SparkSession.builder \
    .appName("TNG_SentimentStreaming") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark session created:", spark.version)

Spark session created: 3.5.1


In [41]:
with open('naive_bayes_model.pickle', 'rb') as f:
    model_data = pickle.load(f)

model_bc = spark.sparkContext.broadcast(model_data)

def predict_sentiment(text):
    if text is None or text.strip() == "":
        return "unknown"
    data = model_bc.value
    vec = data['vectorizer'].transform([text])
    pred = data['model'].predict(vec)
    return str(pred[0])

predict_udf = udf(predict_sentiment, StringType())
print("Model broadcast to Spark executors. Ready to classify streaming data.")

Model broadcast to Spark executors. Ready to classify streaming data.


In [42]:
# Define the JSON schema matching what the producer sends
review_schema = StructType([
    StructField("review_id", StringType(), True),
    StructField("cleaned_text", StringType(), True),
    StructField("true_sentiment", StringType(), True),
    StructField("score", IntegerType(), True),
])

kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", TOPIC_NAME) \
    .option("startingOffsets", "earliest") \
    .load()

# Kafka delivers raw bytes in a 'value' column -- parse it as JSON using our schema
parsed_stream = kafka_stream.select(
    from_json(col("value").cast("string"), review_schema).alias("data")
).select("data.*")

classified_stream = parsed_stream.withColumn("predicted_sentiment", predict_udf(col("cleaned_text")))

def show_batch(batch_df, batch_id):
    count = batch_df.count()
    print(f"\n=== Batch {batch_id}: {count} rows ===")
    if count > 0:
        batch_df.show(20, truncate=False)

query = classified_stream.writeStream \
    .foreachBatch(show_batch) \
    .start()

query.awaitTermination(timeout=60)
query.stop()
print("Streaming query stopped.")


=== Batch 0: 2500 rows ===
+------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+-----+-------------------+
|review_id                           |cleaned_text                                                                                                                                                                                                                                                                                                                |true_sentiment|score|predicted_sentiment|
+------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------

## Part 8: Batch vs. Streaming Performance Comparison

In [43]:
import time
import pandas as pd
import pickle

df_full = pd.read_csv('cleaned_data.csv')
with open('naive_bayes_model.pickle', 'rb') as f:
    model_data = pickle.load(f)

model = model_data['model']
vectorizer = model_data['vectorizer']

N_BATCH = len(df_full)
print(f"Batch-classifying all {N_BATCH} reviews...")

batch_start = time.time()
X_all = vectorizer.transform(df_full['cleaned_text'].astype(str))
batch_predictions = model.predict(X_all)
batch_time = time.time() - batch_start

batch_throughput = N_BATCH / batch_time
print(f"\nBATCH MODE RESULTS:")
print(f"  Total reviews:     {N_BATCH}")
print(f"  Total time:        {batch_time:.3f} seconds")
print(f"  Throughput:        {batch_throughput:.1f} reviews/second")

Batch-classifying all 9979 reviews...

BATCH MODE RESULTS:
  Total reviews:     9979
  Total time:        0.201 seconds
  Throughput:        49685.6 reviews/second


In [44]:
from kafka import KafkaProducer
import json

N_STREAM_COMPARE = 500  # match N_STREAM_DEMO from Part 6 for a fair comparison

producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

stream_compare_start = time.time()

for _, row in df_full.head(N_STREAM_COMPARE).iterrows():
    message = {
        'review_id': str(row['review_id']) + '_cmp',  # suffix to avoid colliding with Part 6's sends
        'cleaned_text': str(row['cleaned_text']),
        'true_sentiment': str(row['sentiment']),
        'score': int(row['score']),
    }
    producer.send(TOPIC_NAME, value=message)
producer.flush()

produce_time = time.time() - stream_compare_start
print(f"Produced {N_STREAM_COMPARE} messages to Kafka in {produce_time:.3f} seconds")
print("Now run the next cell to measure Spark consumption + classification time.")

Produced 500 messages to Kafka in 0.259 seconds
Now run the next cell to measure Spark consumption + classification time.


In [45]:
processed_count = {"n": 0}
consume_start = time.time()
consume_end_time = {"t": None}

def count_batch(batch_df, batch_id):
    n = batch_df.count()
    processed_count["n"] += n
    if processed_count["n"] >= N_STREAM_COMPARE and consume_end_time["t"] is None:
        consume_end_time["t"] = time.time()

kafka_stream_cmp = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", TOPIC_NAME) \
    .option("startingOffsets", "earliest") \
    .load()

parsed_cmp = kafka_stream_cmp.select(
    from_json(col("value").cast("string"), review_schema).alias("data")
).select("data.*").withColumn("predicted_sentiment", predict_udf(col("cleaned_text")))

query_cmp = parsed_cmp.writeStream.foreachBatch(count_batch).start()
query_cmp.awaitTermination(timeout=45)
query_cmp.stop()

consume_time = (consume_end_time["t"] or time.time()) - consume_start
stream_throughput = processed_count["n"] / consume_time if consume_time > 0 else 0

print(f"\nSTREAMING MODE RESULTS:")
print(f"  Reviews processed: {processed_count['n']}")
print(f"  Produce time:      {produce_time:.3f} seconds")
print(f"  Consume+classify:  {consume_time:.3f} seconds")
print(f"  Total time:        {produce_time + consume_time:.3f} seconds")
print(f"  Throughput:        {stream_throughput:.1f} reviews/second")


STREAMING MODE RESULTS:
  Reviews processed: 3000
  Produce time:      0.259 seconds
  Consume+classify:  4.721 seconds
  Total time:        4.980 seconds
  Throughput:        635.5 reviews/second


In [46]:
import pandas as pd

comparison_df = pd.DataFrame([
    {"Mode": "Batch", "Reviews": N_BATCH, "Total Time (s)": round(batch_time, 3), "Throughput (rev/s)": round(batch_throughput, 1)},
    {"Mode": "Streaming (Kafka+Spark)", "Reviews": processed_count["n"], "Total Time (s)": round(produce_time + consume_time, 3), "Throughput (rev/s)": round(stream_throughput, 1)},
])
print(comparison_df.to_string(index=False))
comparison_df.to_csv('batch_vs_streaming_comparison.csv', index=False)
print("\nSaved to batch_vs_streaming_comparison.csv -- download this for your report.")

                   Mode  Reviews  Total Time (s)  Throughput (rev/s)
                  Batch     9979           0.201             49685.6
Streaming (Kafka+Spark)     3000           4.980               635.5

Saved to batch_vs_streaming_comparison.csv -- download this for your report.


## Part 9 (Optional): Write Classified Results to a CSV for Elasticsearch/Kibana

In [47]:
# Batch-classify the full dataset and save as a single export for Elasticsearch loading
df_full['predicted_sentiment'] = batch_predictions
export_cols = ['review_id', 'app_id', 'app_name', 'cleaned_text', 'score', 'sentiment', 'predicted_sentiment', 'review_date'] if 'app_id' in df_full.columns else ['review_id', 'cleaned_text', 'score', 'sentiment', 'predicted_sentiment']
df_full[export_cols].to_csv('classified_reviews_for_elasticsearch.csv', index=False)
print(f"Saved {len(df_full)} classified reviews to classified_reviews_for_elasticsearch.csv")
print("Download this file from the Colab file browser for use in Week 4 (dashboard).")

Saved 9979 classified reviews to classified_reviews_for_elasticsearch.csv
Download this file from the Colab file browser for use in Week 4 (dashboard).


## Cleanup: Stop Kafka

In [48]:
!kafka/bin/kafka-server-stop.sh
print("Kafka broker stopped.")

Kafka broker stopped.
